# Spark Lakehouse — Retail Data Unification
## Silver Layer: Data Cleansing and Unification

### 1. Business Context

The online commerce platform and offline Point-of-Sale (POS) system maintain separate customer, product, and order datasets. Differences in schemas, identifiers, and data formats make it difficult to create a consistent view of retail operations.

The Silver layer processes the raw data ingested by the Bronze layer and prepares standardized datasets for downstream analytics.

### 2. Objectives

- Process the selected daily batch from the Bronze Delta tables.
- Standardize column names and data types across online and offline order datasets.
- Clean selected customer and product attributes.
- Match customer records using normalized mobile-number values.
- Unify product records using SKU identifiers.
- Apply source-preference rules when combining online and offline records.
- Persist the resulting datasets as Silver Delta tables.

### 3. Processing Scope

| Dataset | Processing performed |
|---|---|
| Orders | Aligns online and offline schemas, standardizes selected data types, and combines the datasets |
| Customers | Normalizes mobile-number values, matches records across systems, and combines customer attributes |
| Products | Standardizes selected product attributes and combines records using SKU identifiers |

### 4. Medallion Architecture

- **Bronze:** Source ingestion and metadata capture.
- **Silver:** Cleansing, schema standardization, and cross-system unification.
- **Gold:** Dimensional modelling and analytics-ready datasets.

### 5. Technology Stack

**Apache Spark · PySpark · Databricks · Delta Lake · Spark SQL**

### 6. Expected Outcome

Unified Silver datasets that provide a foundation for further data-quality validation, customer and product dimension construction, and downstream analytics.

**Design principle:** Preserve source lineage while standardizing data for reliable downstream processing.

## 1. Configuration and Batch Selection

The notebook uses a Databricks widget named `file_run_date` to select the source batch for processing.

- The default date is set to yesterday in `YYYYMMDD` format.
- The selected date filters the Bronze tables to process the intended batch.
- The same parameter is reused across the orders, customers, and products processing sections.

This parameter-driven approach makes the notebook reusable across different daily data batches.

In [0]:
from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime("%Y%m%d")

dbutils.widgets.text("file_run_date", yesterday_str)

file_run_date = dbutils.widgets.get("file_run_date")
print(file_run_date)

# from datetime import date, timedelta
# from pyspark.sql.functions import *
# from pyspark.sql.functions import col

# dbutils.widgets.text("file_run_date", "20260208")
# file_run_date = dbutils.widgets.get("file_run_date")

20260207


## 2. Rerun Handling and Batch Replacement

### Purpose

When a daily batch is processed again, previously loaded records for that batch must be handled to avoid duplicate data in the Silver tables.

### Approach

The notebook deletes existing Silver records matching the selected `file_run_date` before processing the batch again.

This allows the selected batch to be reprocessed while retaining records associated with other file dates.

**Operational consideration:** The deletion and subsequent writes are separate operations. If execution fails partway through, the affected batch should be checked before rerunning the notebook.

In [0]:
%sql
delete from `spark-databricks-project-catalog`.silver.orders_unified where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.silver.customers_unified where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.silver.products_unified where file_date = :file_run_date;

num_affected_rows
10004


## 3. Order Data — Schema Standardization and Unification

Online and offline order systems use different column names and representations for several business attributes.

### Processing workflow

1. Read online and offline order data from the Bronze Delta tables.
2. Filter both datasets using the selected `file_run_date`.
3. Map source-specific fields to a common schema, including order ID, order timestamp, customer ID, SKU, quantity, price, discount, and payment mode.
4. Convert selected fields to appropriate data types and trim selected string fields.
5. Populate the store identifier for online orders and retain the offline store identifier where available.
6. Combine the standardized datasets using a Spark DataFrame union.
7. Append the resulting records to `silver.orders_unified`.

### Outcome

A unified order dataset with a common column structure across both retail channels, ready for further customer and product enrichment.

In [0]:
df_online_orders = spark.table("`spark-databricks-project-catalog`.bronze.online_orders")
df_online_orders_filtered = df_online_orders.filter(col("file_date") == file_run_date) #just to read the latest data
#i.e. incremental loading

#df_online_orders_filtered.show(4)

In [0]:
from pyspark.sql.functions import *

df_cleaned_online_orders = df_online_orders_filtered.select(
    trim(col("order_id")).alias("order_id"),
    to_timestamp("order_ts").alias("order_ts"),
    trim(col("customer_id")).alias("customer_id"),
    trim(col("sku_id")).alias("sku_id"),
    col("qty").cast("int").alias("qty"),
    col("list_price").cast("double").alias("list_price"),
    col("discount").cast("double").alias("discount"),
    trim(col("payment_mode")).alias("payment_mode"),
    lit('N/A').alias("store_num"),
    col("file_date"),
    trim(col("source")).alias("source_system"),
    col("ingest_ts").alias("ingestion_time")
)
#df_cleaned_online_orders.show(5)

In [0]:
df_offline_orders = spark.table("`spark-databricks-project-catalog`.bronze.offline_orders")
df_offline_orders_filtered = df_offline_orders.filter(col("file_date") == file_run_date)

#df_offline_orders_filtered.show(5)

In [0]:
from pyspark.sql.functions import *

df_cleaned_offline_orders = df_offline_orders_filtered.select(
    trim(col("txn_id")).alias("order_id"),
    to_timestamp("txn_time").alias("order_ts"),
    trim(col("customer_id")).alias("customer_id"),
    trim(col("sku")).alias("sku_id"),
    col("quantity").cast("int").alias("qty"),
    col("mrp").cast("double").alias("list_price"),
    col("discount").cast("double").alias("discount"),
    trim(col("payment_mode")).alias("payment_mode"),
    col("store_num"),
    col("file_date"),
    trim(col("source")).alias("source_system"),
    col("ingest_ts").alias("ingestion_time")
)  
#df_cleaned_offline_orders.show(5)    

In [0]:
df_orders = df_cleaned_online_orders.union(df_cleaned_offline_orders)
df_orders.write.mode("append") \
         .saveAsTable("`spark-databricks-project-catalog`.silver.orders_unified") 

# df = spark.table("`spark-databricks-project-catalog`.silver.orders_unified")   
# df.show(4)             

## 4. Customer Data — Cross-System Matching

The same customer may be represented separately in the online and offline systems. Their source-specific customer IDs cannot reliably identify the same person across both systems.

### Processing workflow

1. Read the online and offline customer datasets from Bronze.
2. Filter the datasets using the selected file date.
3. Trim the selected mobile-number fields and derive their last ten characters.
4. Remove duplicate mobile-number values within each source dataset.
5. Perform a full outer join using the normalized mobile-number values.
6. Use Spark's `coalesce` function to prefer online customer attributes when available and fall back to offline attributes otherwise.
7. Write the combined records to `silver.customers_unified`.

### Source-preference rule

Where matching mobile-number values exist in both systems, online attributes are selected first for the fields combined using `coalesce`. Offline values provide the fallback for those fields.

### Outcome

A combined customer dataset that brings together online and offline customer attributes and retains source-specific customer identifiers for traceability.

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

df_offline_customers = spark.table("`spark-databricks-project-catalog`.bronze.offline_customers")
df_offline_customers_filtered = df_offline_customers.filter(col("file_date") == file_run_date)
#df_offline_customers_filtered.show(4)

phone_trimmed = trim(col("phone"))

df_offline_customers_filtered_clean = df_offline_customers_filtered \
                             .withColumn("phone", substring(phone_trimmed, length(phone_trimmed) - 9, 10)) \
                             .withColumn("name", trim(col("name")))                           

df_offline_customers_filtered_clean = df_offline_customers_filtered_clean.dropDuplicates(["phone"]) #- not required as the data is already deduped  
#df_offline_customers_filtered_clean.show(4)


In [0]:
df_online_customers = spark.table("`spark-databricks-project-catalog`.bronze.online_customers")
df_online_customers_filtered = df_online_customers.filter(col("file_date") == file_run_date)
#df_online_customers_filtered.show(4)
phone_trimmed = trim(col("mobile_number"))

df_online_customers_filtered_clean = df_online_customers_filtered \
                         .withColumn("mobile_number", substring(phone_trimmed, length(phone_trimmed) - 9, 10)) \
                         .withColumn("full name", trim(col("full_name")))      

df_online_customers_filtered_clean = df_online_customers_filtered_clean.dropDuplicates(["mobile_number"])
#df_online_customers_filtered_clean.show(4)



In [0]:
joined = df_online_customers_filtered_clean.alias("on") \
              .join(df_offline_customers_filtered_clean.alias("off"), \
               col("on.mobile_number") == col("off.phone") \
                   ,"full_outer")

#joined.show(5)                

In [0]:
from pyspark.sql.functions import col, coalesce

final_df = joined.select(
    coalesce(col("on.full_name"), col("off.name")).alias("full_name"),
    coalesce(col("on.email"), col("off.email")).alias("email"),
    coalesce(col("on.mobile_number"), col("off.phone")).alias("mobile_number"),
    coalesce(col("on.city"), col("off.city")).alias("city"),
    coalesce(col("on.state"), col("off.state")).alias("state"),
    coalesce(col("on.zipcode"), col("off.pincode")).alias("zipcode"),
    col("on.signup_date").cast('date').alias("signup_date"),
    #col("off.store_id").alias("store_id") not taken while unifying customer's data
    col("off.cust_id").alias("offline_customer_id"),
    col("on.customer_id").alias("online_customer_id"),
    coalesce(col("on.file_date"), col("off.file_date")).alias("file_date"),
    coalesce(col("on.source"), col("off.source")).alias("source"),
    coalesce(col("on.ingest_ts"), col("off.ingest_ts")).alias("ingest_ts")
)
#final_df.show(5)

In [0]:
final_df.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.silver.customers_unified")

## 5. Product Data — SKU-Based Unification

Online and offline product datasets may use different column names for product attributes. The SKU identifier provides the common key used to match product records across the two systems.

### Processing workflow

1. Read the online and offline product datasets from Bronze.
2. Filter both datasets using the selected file date.
3. Remove duplicate SKU values within each source dataset.
4. Trim the selected product-name fields.
5. Perform a full outer join using the online SKU and offline SKU columns.
6. Combine product attributes using `coalesce`, preferring online values when available.
7. Retain selected source-specific attributes, including currency, active status, and tax rate.
8. Append the combined records to `silver.products_unified`.

### Outcome

A unified product dataset that combines records across the online and offline retail systems and provides a common product structure for downstream processing.

In [0]:
from pyspark.sql.functions import col, coalesce

df_online_products = spark.table("`spark-databricks-project-catalog`.bronze.online_products")
df_online_products_filtered = df_online_products.filter(col("file_date") == file_run_date)

df_online_products_filtered =df_online_products_filtered.dropDuplicates(["sku_id"])

df_online_products_filtered = df_online_products_filtered.withColumn("product_name", trim(col("product_name")) )

In [0]:
df_offline_products = spark.table("`spark-databricks-project-catalog`.bronze.offline_products")
df_offline_products_filtered = df_offline_products.filter(col("file_date") == file_run_date)

df_offline_products_filtered = df_offline_products_filtered.dropDuplicates(["sku"])
df_offline_products_filtered = df_offline_products_filtered.withColumn("name", trim(col("name")))

In [0]:
from pyspark.sql.functions import col, coalesce

joined_prod = df_online_products_filtered.alias("on") \
              .join(df_offline_products_filtered.alias("off") \
              , col("on.sku_id") == col("off.sku") \
              , "full_outer"
            )
#joined_prod.show(5)       

In [0]:
from pyspark.sql.functions import col, coalesce

final_products_df = joined_prod.select(
    coalesce(col("on.sku_id"), col("off.sku")).alias("sku_id"),
    coalesce(col("on.product_name"), col("off.name")).alias("product_name"),
    coalesce(col("on.category"), col("off.category_name")).alias("category"),
    coalesce(col("on.brand"), col("off.brand_name")).alias("brand"),
    coalesce(col("on.price"), col("off.mrp_price")).cast('float').alias("price"),

    #only in online data
    col("on.currency").alias("currency"),
    col("on.is_active").alias("is_active"),

    #only in offline data
    col("off.tax_rate").cast('float').alias("tax_rate"),
    
    coalesce(col("on.file_date"), col("off.file_date")).alias("file_date"),
    coalesce(col("on.source"), col("off.source")).alias("source"),
    coalesce(col("on.ingest_ts"), col("off.ingest_ts")).alias("ingest_ts")   
)
#final_products_df.show(4)
final_products_df.write.mode("append").saveAsTable("`spark-databricks-project-catalog`.silver.products_unified")

## 6. Silver Layer — Output Summary

### Target tables

| Target table | Purpose |
|---|---|
| `silver.orders_unified` | Combined online and offline order records |
| `silver.customers_unified` | Combined customer records matched using mobile-number values |
| `silver.products_unified` | Combined product records matched using SKU identifiers |

### Key Data Engineering Concepts Demonstrated

- Parameter-driven batch processing.
- Incremental date filtering.
- Spark DataFrame transformations and column mapping.
- Data-type standardization.
- Deduplication within source datasets.
- Full outer joins for cross-system record matching.
- Source-preference rules using `coalesce`.
- Delta table writes and batch rerun handling.

### Next Stage

The Gold layer will build analytics-ready dimensions and fact tables from the processed data, using business keys, surrogate keys, and historical tracking where required by the project specification.